# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an interactive workflow for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

print(f"Dataset version: {metadata.version}")

## 2. Data Overview
Review available record sets, their fields, and their `@id` values.

In [ ]:
# List the record sets, their @id, and their fields
print('Record sets in this dataset:')
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"  RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id})  | dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets and field references use their `@id`.

In [ ]:
# Extract data from all record sets using their @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
print(f"Record sets to extract: {record_set_ids}")

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set.id))
    if len(records) == 0:
        print(f"No records found for {record_set.id}")
        continue
    dataframes[record_set.id] = pd.DataFrame(records)
    print(f"Loaded RecordSet: {record_set.id}  |  {len(dataframes[record_set.id])} records  | Columns: {list(dataframes[record_set.id].columns)}")

# Display the columns in the main record set (first in the list)
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]
print(f"Fields (@id) in {main_record_set_id}:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform common data processing steps such as filtering, normalization, and grouping. All columns are referenced by their `@id` as required by the Croissant schema.

In [ ]:
# Use known numeric fields for EDA. Update these @id values as per dataset contents.
# For demonstration, use '@id' for PHQ-9 total score, which is likely named 'phq9_total_score' or similar.
# Use the correct field name as per printout in previous cell!

# For illustration, let's assume phq9_total_score's @id is 'phq9_total_score'
numeric_field = 'phq9_total_score'

if numeric_field not in main_df.columns:
    # Try GAD-7 if PHQ-9 is not present
    numeric_field = 'gad7_total_score'
if numeric_field not in main_df.columns:
    # Try PSQ
    numeric_field = 'psq_total_score'
if numeric_field not in main_df.columns:
    numeric_field = main_df.select_dtypes('number').columns[0] if len(main_df.select_dtypes('number').columns) > 0 else main_df.columns[0]

print(f"Numeric field selected for EDA: {numeric_field}")

# Remove missing or non-numeric
filtered_main_df = main_df.copy()
filtered_main_df = filtered_main_df[pd.to_numeric(filtered_main_df[numeric_field], errors='coerce').notna()]
filtered_main_df[numeric_field] = filtered_main_df[numeric_field].astype(float)

# Filter records with high scores (e.g., threshold=10)
threshold = 10
filtered_df = filtered_main_df[filtered_main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a key attribute (such as 'level_of_education' if present by @id)
group_field = 'level_of_education'
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).agg({numeric_field: 'mean', f'{numeric_field}_normalized': 'mean'})
    print(f"Grouped data by {group_field} (mean values):")
    print(grouped_df)
else:
    print(f"Group field '{group_field}' not found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All visualizations use field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution histogram for the selected numeric field
plt.figure(figsize=(7, 4))
sns.histplot(main_df[numeric_field].dropna(), bins=10, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by level_of_education if it exists
if 'level_of_education' in main_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x='level_of_education', y=numeric_field, data=main_df)
    plt.title(f'{numeric_field} by level_of_education')
    plt.xticks(rotation=40)
    plt.ylabel(numeric_field)
    plt.xlabel('level_of_education')
    plt.show()
# Pairplot with a few key demographic fields if present
demo_fields = [field for field in ['age', 'gender', 'marital_status'] if field in main_df.columns]
if len(demo_fields) > 0:
    sns.pairplot(main_df, vars=[numeric_field], hue=demo_fields[0])
    plt.show()

## 6. Conclusion
Summarize key findings from the dataset exploration.

Through this notebook, we loaded the Kilifi County FAIR² Mental Health Survey dataset using the Croissant schema and `mlcroissant`, explored the structure via `@id` fields, extracted structured data, performed basic EDA steps such as filtering and normalization, grouped data by key demographic variables, and visualized distributions. You can now proceed to more advanced modeling or analysis, continuing to reference all fields and record sets by their Croissant `@id` for full reproducibility and standards compliance.